# certainty

answer w/o retrieval
answer w/ vector search
compare to baseline

In [1]:
%pip install ollama numpy pandas openpyxl

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os, json, re, time, math, hashlib, textwrap
from pathlib import Path
from dataclasses import dataclass, field, asdict, replace
from typing import List, Dict, Any, Optional
from collections import Counter
from itertools import product
import ollama
import numpy as np
import pandas as pd

OLLAMA_URL  = "http://localhost:11528"
AGENT_MODEL = "gpt-oss:120b"
EMBED_MODEL = "nomic-embed-text"
CACHE_DIR  = Path("tree_cache"); TREE_FILE = CACHE_DIR / "corpus_tree.json"
QUESTIONS_FILE = "eval_questions.xlsx"
MAX_EVIDENCE  = 12
SCORE_MODE    = "judge"
EXCLUDE_WORDS = ("which",)
KEEP_ALIVE    = "30m"
print(f"agent {AGENT_MODEL}; embed {EMBED_MODEL}; questions {QUESTIONS_FILE}")

agent gpt-oss:120b; embed nomic-embed-text; questions eval_questions.xlsx


In [3]:
client = ollama.Client(host=OLLAMA_URL, timeout=600)

def _model_names(r):
    raw = r.get("models", []) if hasattr(r, "get") else getattr(r, "models", [])
    out=[]
    for m in raw:
        n = getattr(m,"model",None) or getattr(m,"name",None)
        if n is None and isinstance(m,dict): n=m.get("model") or m.get("name")
        if n: out.append(n)
    return out

def _wait_until_ready():
    announced=False
    while True:
        try:
            names=_model_names(client.list())
            if any(AGENT_MODEL in n for n in names):
                if not any(EMBED_MODEL in n for n in names):
                    print(f"note: {EMBED_MODEL} not present; the ceiling needs it for similarity")
                print(f"ollama ok; {AGENT_MODEL} is loaded"); return
            reason=f"{AGENT_MODEL} not loaded yet"
        except Exception as e:
            reason=f"server unreachable; {type(e).__name__}: {e}"
        if not announced:
            print(f"waiting for ollama, {reason}; rechecking every 10s and wont stop"); announced=True
        time.sleep(10)

def llm(prompt, cfg, counter, num_predict=None, temperature=0, think=None):
    use_think = cfg.thinking if think is None else think
    opts={"temperature":temperature, "num_predict": num_predict or cfg.decision_cap}
    attempt=0; pass_think=True
    while True:
        kw=dict(model=AGENT_MODEL, messages=[{"role":"user","content":prompt}],
                options=opts, keep_alive=KEEP_ALIVE)
        if pass_think: kw["think"]=use_think
        try:
            r=client.chat(**kw)
            counter.calls+=1
            try: counter.in_tok  += int(r["prompt_eval_count"] or 0)
            except Exception: pass
            try: counter.out_tok += int(r["eval_count"] or 0)
            except Exception: pass
            txt=(r["message"]["content"] or "").strip()
            if not txt:
                try: txt=(r["message"]["thinking"] or "").strip()
                except Exception: pass
            return txt
        except TypeError:
            pass_think=False
        except Exception as e:
            attempt+=1
            if attempt==1 or attempt%5==0: print(f"[waiting for ollama] {type(e).__name__}: {e}; retrying")
            time.sleep(min(60, 5*2**min(attempt-1,4)))

def embed(text, counter):
    try:
        r=client.embeddings(model=EMBED_MODEL, prompt=text or " ")
        try: counter.in_tok += int(r.get("prompt_eval_count",0) or 0)
        except Exception: pass
        return r["embedding"]
    except Exception:
        return None

_wait_until_ready()
print("llm and embed helpers ready")

ollama ok; gpt-oss:120b is loaded
llm and embed helpers ready


In [4]:
import re, math, hashlib
from dataclasses import dataclass, field, asdict
from typing import List, Dict, Any, Optional
from collections import Counter
import numpy as np


@dataclass
class TreeNode:
    node_id:str; node_type:str; name:str; path:str; summary:str
    content:str=""; children:List["TreeNode"]=field(default_factory=list)
    metadata:Dict[str,Any]=field(default_factory=dict)

    @classmethod
    def from_dict(cls,d):
        n=cls(node_id=d["node_id"],node_type=d["node_type"],name=d["name"],
              path=d.get("path",""),summary=d.get("summary",""),
              content=d.get("content",""),metadata=d.get("metadata",{}))
        n.children=[cls.from_dict(c) for c in d.get("children",[])]
        return n

    def is_leaf(self): return self.node_type=="chunk"
    def count_leaves(self): return 1 if self.is_leaf() else sum(c.count_leaves() for c in self.children)


@dataclass
class Question:
    qid:str; stem:str; options:Dict[str,str]; answer:str; difficulty:str=""


@dataclass(frozen=True)
class Config:
    strategy:str="cluster"          
    working_memory:bool=True        
    backtracking:bool=True          
    thinking:bool=False             
    group_summary:str="heuristic"   
    max_branch:int=6                
    decision_cap:int=512            
    answer_cap:int=8                
    nav_includes_options:bool=False 
    breadcrumb:bool=False           
    vote_samples:int=1              
    max_steps:int=24                



def config_name(cfg):
    base=Config()
    diffs=[f"{k}={getattr(cfg,k)}" for k in cfg.__dataclass_fields__ if getattr(cfg,k)!=getattr(base,k)]
    return "baseline" if not diffs else ", ".join(diffs)

def config_key(cfg):
    return hashlib.md5(repr(asdict(cfg)).encode()).hexdigest()[:12]


class Counters:
    def __init__(self): self.in_tok=0; self.out_tok=0; self.calls=0


def clip(t,n):
    t=re.sub(r"\s+"," ",t or "").strip()
    return t if len(t)<=n else t[:n]+" …"



_nav_cache={}; _embed_cache={}; _gsum_cache={}

def reset_vcaches():
    _nav_cache.clear(); _gsum_cache.clear()   

def _embed_node(node,counter):
    if node.node_id in _embed_cache: return _embed_cache[node.node_id]
    v=embed((node.name+". "+(node.summary or ""))[:2000],counter)
    _embed_cache[node.node_id]=v; return v

def _kmeans(vectors,k,iters=25,seed=0):
    X=np.asarray(vectors,dtype=float); n=len(X); k=max(1,min(k,n))
    X=X/np.clip(np.linalg.norm(X,axis=1,keepdims=True),1e-9,None)
    rng=np.random.default_rng(seed); C=X[rng.choice(n,size=k,replace=False)].copy()
    labels=np.full(n,-1)
    for _ in range(iters):
        d=((X[:,None,:]-C[None,:,:])**2).sum(-1); new=d.argmin(1)
        if np.array_equal(new,labels): break
        labels=new
        for j in range(k):
            pts=X[labels==j]; C[j]=pts.mean(0) if len(pts) else X[rng.integers(n)]
    return labels.tolist()

def _group_summary(members,label,cfg,counter):
    key=(label,cfg.group_summary,tuple(m.node_id for m in members))
    if key in _gsum_cache: return _gsum_cache[key]
    if cfg.group_summary=="llm":
        joined="\n".join(f"- {m.summary}" for m in members)
        prompt=("in 2-4 sentences say what this group called '"+label+"' covers so an agent can "
                "decide whether to explore it; list the main topics. respond with only the text.\n\n"+joined)
        s=llm(prompt,cfg,counter,num_predict=300)
    else:
        lines=[f"- {m.name}: {clip(m.summary,160)}" for m in members[:cfg.max_branch]]
        more=f"\n- …and {len(members)-cfg.max_branch} more" if len(members)>cfg.max_branch else ""
        s=f"a group of {len(members)} related items:\n"+"\n".join(lines)+more
    _gsum_cache[key]=s; return s

def _vid(pid,tag): return "v"+hashlib.md5(f"{pid}|{tag}".encode()).hexdigest()[:11]

def _make_vgroup(parent,members,idx,cfg,counter):
    return TreeNode(node_id=_vid(parent.node_id,f"{cfg.strategy}:{cfg.max_branch}:{idx}"),
                    node_type="vgroup",name=f"[group {idx+1} · {len(members)} items]",path="",
                    summary=_group_summary(members,f"{parent.name} group {idx+1}",cfg,counter),
                    children=list(members),metadata={"virtual":True})

def _chunk_groups(parent,kids,cfg,counter):
    size=math.ceil(len(kids)/cfg.max_branch)              
    groups=[kids[i:i+size] for i in range(0,len(kids),size)]
    return [_make_vgroup(parent,g,i,cfg,counter) for i,g in enumerate(groups)]

def _cluster_groups(parent,kids,cfg,counter):
    vecs=[_embed_node(k,counter) for k in kids]
    if any(v is None for v in vecs): return _chunk_groups(parent,kids,cfg,counter)  
    k=max(2,min(cfg.max_branch,math.ceil(len(kids)/max(2,cfg.max_branch-2))))
    labels=_kmeans(vecs,k); buckets={}
    for kid,lab in zip(kids,labels): buckets.setdefault(lab,[]).append(kid)
    if len(buckets)<2 or max(len(v) for v in buckets.values())==len(kids):
        return _chunk_groups(parent,kids,cfg,counter)     
    ordered=[buckets[lab] for lab in sorted(buckets)]
    return [_make_vgroup(parent,m,i,cfg,counter) for i,m in enumerate(ordered)]

def get_nav_children(node,cfg,counter):
    key=(node.node_id,cfg.strategy,cfg.max_branch,cfg.group_summary)
    if key in _nav_cache: return _nav_cache[key]
    kids=node.children
    if cfg.strategy=="none" or len(kids)<=cfg.max_branch: res=list(kids)
    elif cfg.strategy=="chunk": res=_chunk_groups(node,kids,cfg,counter)
    elif cfg.strategy=="cluster": res=_cluster_groups(node,kids,cfg,counter)
    else: res=list(kids)
    _nav_cache[key]=res; return res


import json as _json

def _add_memory(mem,fact,cap=20):
    fact=clip(fact,500)
    if fact and fact.lower() not in ("","none","n/a") and fact not in mem:
        mem.append(fact); del mem[:-cap]

def _parse_decision(raw,n_opts,wm):
    s=re.sub(r"^```(?:json)?|```$","",raw.strip(),flags=re.M).strip()
    m=re.search(r"\{.*\}",s,flags=re.S)
    if m:
        try:
            d=_json.loads(m.group(0)); act=str(d.get("action","")).lower().strip()
            if act not in ("descend","answer","backtrack"): act="descend" if n_opts else "answer"
            ci=d.get("child",None)
            try: ci=int(ci)
            except (TypeError,ValueError): ci=None
            return {"action":act,"child":ci,"remember":(d.get("remember") or "").strip() if wm else ""}
        except Exception: pass
    return {"action":"descend" if n_opts else "backtrack","child":0 if n_opts else None,"remember":""}

def _ask(query,node,options,memory,cfg,counter,can_back):
    opts_txt="\n".join(f"[{i}] {o.name} — {clip(o.summary,360)}" for i,o in enumerate(options)) or "(no options here)"
    acts=(['"descend"'] if options else [])+['"answer"']+(['"backtrack"'] if can_back else [])
    mem_block=""; remember_field=""
    if cfg.working_memory:
        mem_block="WORKING MEMORY (facts gathered so far):\n"+("\n".join("- "+m for m in memory) or "(empty)")+"\n\n"
        remember_field='"remember": "<a useful fact from the CURRENT summary to keep, else empty>", '
    prompt=(
        "you are an agent navigating a tree of document summaries to answer a question. "
        "you see only summaries and move one node at a time.\n\n"
        f"QUESTION: {query}\n\n{mem_block}"
        f"CURRENT NODE: {node.name} [{node.node_type}]\nCURRENT SUMMARY: {clip(node.summary,900)}\n\n"
        f"CHILD OPTIONS:\n{opts_txt}\n\n"
        f"choose ONE action ({', '.join(acts)}). reply with ONLY json:\n"
        '{"reasoning":"<1 sentence>", '+remember_field+
        '"action":"descend|answer|backtrack", "child":<index or null>}\n'
        "descend into the option most likely to lead to the answer; answer if you have enough; "
        "backtrack if none of these are relevant.")
    cap=cfg.decision_cap if not cfg.thinking else max(cfg.decision_cap,1024)
    return _parse_decision(llm(prompt,cfg,counter,num_predict=cap),len(options),cfg.working_memory)

SCORE_MODE = globals().get("SCORE_MODE","judge")  

def _short(n):
    if n.metadata.get("virtual"): return "[grp]"
    return (n.name or "?")[:22]

def _src(n): return n.metadata.get("source_file") or n.path or n.name

def run_agent(q,cfg,counter,trace=True,live=False):
    ink=(lambda s: print(s,end="",flush=True)) if live else (lambda s: None)
    nav_q=q.stem                                   
    if cfg.nav_includes_options:
        nav_q+="\noptions: "+"; ".join(f"{L}) {q.options[L]}" for L in "ABCD")
    memory=[]; evidence=[]; seen=set(); visited={ROOT.node_id}; crumbs=[]
    trail=["root"]; backtracks=0
    stack=[{"node":ROOT,"options":get_nav_children(ROOT,cfg,counter),"tried":set()}]
    ink("  path: root")
    steps=0
    while stack and steps<cfg.max_steps:
        steps+=1; fr=stack[-1]; node=fr["node"]
        present=[(i,o) for i,o in enumerate(fr["options"]) if i not in fr["tried"] and o.node_id not in visited]
        opts=[o for _,o in present]
        can_back=cfg.backtracking and len(stack)>1
        dec=_ask(nav_q,node,opts,memory,cfg,counter,can_back)
        if cfg.working_memory and dec["remember"]: _add_memory(memory,dec["remember"])
        if cfg.breadcrumb: crumbs.append(node.summary)
        act=dec["action"]
        if act=="answer" and not evidence and opts: act="descend"   
        if act=="descend" and not opts:                             
            act="backtrack" if can_back else "answer"
        if act=="answer" or (act=="backtrack" and not can_back):    
            if node.node_id not in seen: evidence.append(node); seen.add(node.node_id)
            ink(" ⇒ answer"); break
        if act=="backtrack":
            popped=stack.pop(); parent=stack[-1]; backtracks+=1; trail.append("↩"); ink(" ↩")
            for i,o in enumerate(parent["options"]):
                if o.node_id==popped["node"].node_id: parent["tried"].add(i); break
            continue
        ci=dec["child"]
        if ci is None or not (0<=ci<len(opts)): ci=0    
        child=opts[ci]; visited.add(child.node_id)
        child_opts=get_nav_children(child,cfg,counter)
        if child.is_leaf() or not child_opts:           
            if child.node_id not in seen: evidence.append(child); seen.add(child.node_id)
            if cfg.working_memory and child.content: _add_memory(memory,clip(child.content,600))
            trail.append(_short(child)+"*"); ink(f" → {child.name}✓")
            for i,o in enumerate(fr["options"]):
                if o.node_id==child.node_id: fr["tried"].add(i); break
            continue
        trail.append(_short(child)); ink(f" → {child.name}")
        stack.append({"node":child,"options":child_opts,"tried":set()})
    if live: print()                                    
    if SCORE_MODE=="letter":
        response=_answer_letter(q,memory,evidence,crumbs,cfg,counter)
    else:
        response=_answer_response(q,memory,evidence,crumbs,cfg,counter)
    return {"response":response,"evidence":evidence,"steps":steps,
            "backtracks":backtracks,"path":" › ".join(trail)}

def _parse_letter(text):
    t=(text or "").strip().upper()
    if not t: return ""
    cues=re.findall(r"ANSWER[^ABCD]{0,8}?\b([ABCD])\b",t)   
    if cues: return cues[-1]
    m=re.search(r"\b([ABCD])\b",t) or re.search(r"([ABCD])",t)
    return m.group(1) if m else ""

def _answer_letter(q,memory,evidence,crumbs,cfg,counter):
    opts="\n".join(f"{L}. {q.options[L]}" for L in "ABCD")
    parts=[]
    if cfg.working_memory and memory: parts.append("notes you gathered:\n"+"\n".join("- "+m for m in memory))
    if cfg.breadcrumb and crumbs: parts.append("summaries along your path:\n"+"\n".join("- "+clip(c,200) for c in crumbs[-8:]))
    if evidence:
        ev="\n\n".join(f"[{e.metadata.get('source_file') or e.path or e.name}] {clip(e.content or e.summary,1200)}"
                       for e in evidence[:MAX_EVIDENCE])
        parts.append("evidence:\n"+ev)
    ctx="\n\n".join(parts) or "(no context gathered)"
    prompt=(f"answer the multiple choice question using the gathered information.\n\n"
            f"QUESTION: {q.stem}\nOPTIONS:\n{opts}\n\n{ctx}\n\n"
            "respond with ONLY the single letter of the best option: A, B, C, or D.")
    cap=1024 if cfg.thinking else 64
    if cfg.vote_samples<=1:
        return _parse_letter(llm(prompt,cfg,counter,num_predict=cap,temperature=0))
    votes=[_parse_letter(llm(prompt,cfg,counter,num_predict=cap,temperature=0.7)) for _ in range(cfg.vote_samples)]
    votes=[v for v in votes if v]
    return Counter(votes).most_common(1)[0][0] if votes else ""

def _answer_response(q,memory,evidence,crumbs,cfg,counter):
    parts=[]
    if cfg.working_memory and memory: parts.append("notes you gathered:\n"+"\n".join("- "+m for m in memory))
    if cfg.breadcrumb and crumbs: parts.append("summaries along your path:\n"+"\n".join("- "+clip(c,200) for c in crumbs[-8:]))
    if evidence:
        ev="\n\n".join(f"[{e.metadata.get('source_file') or e.path or e.name}] {clip(e.content or e.summary,1200)}"
                       for e in evidence[:MAX_EVIDENCE])
        parts.append("evidence:\n"+ev)
    ctx="\n\n".join(parts) or "(no context gathered)"
    prompt=(f"answer the question using only the gathered information; be specific.\n\n"
            f"QUESTION: {q.stem}\n\n{ctx}\n\n"
            "give the answer in 1-3 sentences, and cite the source file in brackets if relevant.")
    return llm(prompt,cfg,counter,num_predict=1024 if cfg.thinking else 320,temperature=0)

_JUDGE_CFG=Config()     

def _parse_judge(raw):
    score=0.0; reason=""
    m=re.search(r"\{.*\}",raw or "",flags=re.S)
    if m:
        try:
            d=_json.loads(m.group(0)); score=float(d.get("score",0)); reason=str(d.get("reason","")).strip()
        except Exception: pass
    if not reason:                      
        mm=re.search(r"(0?\.\d+|0|1(?:\.0+)?)",raw or "")
        if mm: score=float(mm.group(1))
    return max(0.0,min(1.0,score)), reason

def judge_score(q,response,counter):
    opts="\n".join(f"{L}. {q.options[L]}" for L in "ABCD")
    gold=q.options.get(q.answer,"")
    prompt=("you are a fair grader. a student answered an open question in their own words and could not see the "
            "choices. the multiple choice version below has the correct option marked and that option is the "
            "ground truth.\n\n"
            f"QUESTION: {q.stem}\nOPTIONS:\n{opts}\nCORRECT OPTION: {q.answer}. {gold}\n\n"
            f"STUDENT RESPONSE:\n{response or '(empty)'}\n\n"
            "grade from 0.0 to 1.0 how well the response matches the MEANING of the correct option. judge by "
            "meaning not wording, so give full or near full credit when the response conveys the same idea even in "
            "different words, partial credit when it is incomplete or hedged, and low credit only when it matches a "
            "different option or is irrelevant or wrong. reply with ONLY json:\n"
            '{"score": <number 0.0 to 1.0>, "reason": "<one concise sentence comparing the response to the correct option>"}')
    return _parse_judge(llm(prompt,_JUDGE_CFG,counter,num_predict=400,temperature=0))

def judge_evidence(q,evidence,counter):
    gold=q.options.get(q.answer,"")
    ev="\n\n".join(f"[{e.metadata.get('source_file') or e.path or e.name}] {clip(e.content or e.summary,1500)}"
                   for e in evidence[:MAX_EVIDENCE]) or "(nothing was retrieved)"
    prompt=("you are checking whether a retrieval system fetched the right information, not whether anyone answered. "
            "below is the correct answer to a question and the text the system retrieved.\n\n"
            f"QUESTION: {q.stem}\nCORRECT ANSWER: {q.answer}. {gold}\n\n"
            f"RETRIEVED TEXT:\n{ev}\n\n"
            "rate from 0.0 to 1.0 how well the retrieved text CONTAINS the information needed to reach the correct "
            "answer, whether or not it is phrased as the answer. 1.0 means the needed facts are clearly present, "
            "0.0 means they are absent. reply with ONLY json:\n"
            '{"score": <number 0.0 to 1.0>, "reason": "<one concise sentence>"}')
    return _parse_judge(llm(prompt,_JUDGE_CFG,counter,num_predict=400,temperature=0))

print("eval core ready; configs, virtual subfolders and the agent are defined")

eval core ready; configs, virtual subfolders and the agent are defined


In [5]:
if not TREE_FILE.exists():
    raise SystemExit(f"tree not found at {TREE_FILE.resolve()}; build it first with prototype 5")
ROOT = TreeNode.from_dict(json.loads(TREE_FILE.read_text(encoding="utf-8")))
print(f"loaded tree {ROOT.name}; {ROOT.count_leaves()} leaves and {len(ROOT.children)} top level children")

loaded tree folders; 95458 leaves and 10 top level children


In [6]:
import json, time, re
from pathlib import Path
import pandas as pd
try:
    from tqdm.auto import tqdm as _tqdm          
except Exception:
    _tqdm=None                                    

def _blank(v):
    if v is None: return True
    s=str(v).strip(); return s=="" or s.lower()=="nan"

def _parse_options_cell(text):
    opts={}
    for line in re.split(r"[\r\n;]+", str(text)):
        mm=re.match(r"^\s*\(?\s*([A-Da-d])\s*[).:\-\u2013]\s*(.+)$", line.strip())
        if mm and mm.group(1).upper() not in opts: opts[mm.group(1).upper()]=mm.group(2).strip()
    return opts

def _match(df):
    norm={re.sub(r"[^a-z]","",str(c).lower()):c for c in df.columns}
    def col(*names):
        for n in names:
            if n in norm: return norm[n]
        return None
    return dict(
        q=col("question","q","prompt","stem"),
        ans=col("correctanswer","answer","correct","gold","label","key"),
        idc=col("number","id","qid","index"),
        diff=col("difficulty","level"),
        combined=col("mcoptions","options","choices","mcq","multiplechoiceoptions","mcoptionsabcd"),
        A=col("a","optiona","choicea"), B=col("b","optionb","choiceb"),
        C=col("c","optionc","choicec"), D=col("d","optiond","choiced"))

def _find_header(raw):
    best,score=0,-1
    for r in range(min(8,len(raw))):
        cells=[re.sub(r"[^a-z]","",str(x).lower()) for x in raw.iloc[r].tolist()]
        s=sum(any(k in c for k in ("question","answer","option","number","choice")) for c in cells)
        if s>score: score,best=s,r
    return best

def _prep(raw):
    hr=_find_header(raw)
    df=raw.iloc[hr+1:].copy(); df.columns=[str(c) for c in raw.iloc[hr].tolist()]
    return df.reset_index(drop=True)

def _rows_from_df(df, sheet, out, seen):
    m=_match(df)
    has_split=all(m[k] for k in ("A","B","C","D"))
    if not (m["q"] and m["ans"] and (has_split or m["combined"])):
        print(f"  skip sheet '{sheet}'; columns found were {list(df.columns)}"); return

    def add(qid,stem,opts,ansv,diffv):
        if _blank(stem) or any(L not in opts or _blank(opts[L]) for L in "ABCD") or _blank(ansv):
            return False
        am=re.search(r"[A-Da-d]",str(ansv))
        if not am: return False
        qid=str(qid).strip() if not _blank(qid) else f"{sheet}_{len(out)+1}"
        while qid in seen: qid+="_x"                       
        seen.add(qid)
        out.append(Question(qid,str(stem).strip(),{L:opts[L] for L in "ABCD"},
                            am.group(0).upper(),"" if _blank(diffv) else str(diffv).strip()))
        return True

    kept=0
    if has_split:
        for _,r in df.iterrows():
            opts={L:str(r[m[L]]).strip() for L in "ABCD"}
            kept+=add(r[m["idc"]] if m["idc"] else None, r[m["q"]], opts, r[m["ans"]],
                      r[m["diff"]] if m["diff"] else None)
    else:
        keycol=m["idc"] or m["q"]                          
        group=df[keycol].apply(lambda x: not _blank(x)).cumsum()
        for _,sub in df.groupby(group):
            def first(c):                                 
                for v in sub[c].tolist():
                    if not _blank(v): return v
                return None
            lines="\n".join(str(v) for v in sub[m["combined"]].tolist() if not _blank(v))
            opts=_parse_options_cell(lines)
            kept+=add(first(m["idc"]) if m["idc"] else None, first(m["q"]), opts, first(m["ans"]),
                      first(m["diff"]) if m["diff"] else None)
    print(f"  sheet '{sheet}': kept {kept}")

def load_questions(path, exclude_words=("which",)):
    p=Path(path); ext=p.suffix.lower(); out=[]; seen=set()
    if ext in (".xlsx",".xls"):
        for name,raw in pd.read_excel(p, sheet_name=None, header=None).items():   
            _rows_from_df(_prep(raw), name, out, seen)
    elif ext==".tsv":
        _rows_from_df(_prep(pd.read_csv(p,sep="\t",header=None,dtype=object)), "tsv", out, seen)
    else:
        _rows_from_df(_prep(pd.read_csv(p,header=None,dtype=object)), "csv", out, seen)
    if exclude_words:                                  
        ban=[w.lower() for w in exclude_words]
        before=len(out)
        out=[q for q in out if not any(w in q.stem.lower() for w in ban)]
        if before-len(out): print(f"excluded {before-len(out)} questions containing {list(exclude_words)}")
    if not out:
        raise ValueError("no valid questions parsed; need question, options and a correct answer letter")
    print(f"parsed {len(out)} questions total from {p.name}")
    return out


EVAL_CACHE_DIR=Path("eval_cache"); RESULTS_FILE=EVAL_CACHE_DIR/"results.json"
CACHE={}

def load_cache():
    global CACHE
    if RESULTS_FILE.exists():
        CACHE=json.loads(RESULTS_FILE.read_text())
    print(f"results cache: {len(CACHE)} stored runs")

def save_cache():
    EVAL_CACHE_DIR.mkdir(exist_ok=True)
    tmp=RESULTS_FILE.with_suffix(".tmp")          
    tmp.write_text(json.dumps(CACHE)); tmp.replace(RESULTS_FILE)

class _PlainBar:
    def __init__(self,total,initial,desc): self.t=total; self.n=initial; self.d=desc
    def update(self,k=1): self.n+=k; print(f"\r{self.d} {self.n}/{self.t}",end="",flush=True)
    def set_postfix(self,**k): pass
    def set_description(self,d): self.d=d
    def close(self): print()

def _bar(total,initial,desc):
    if _tqdm is not None:
        return _tqdm(total=total,initial=initial,unit="q",desc=desc,dynamic_ncols=True)
    return _PlainBar(total,initial,desc)

import random as _random
from dataclasses import replace as _replace

def _eval_one(cfg,q,live=False):
    c=Counters(); t0=time.perf_counter()
    res=run_agent(q,cfg,c,trace=True,live=live); dt=time.perf_counter()-t0
    if SCORE_MODE=="letter":
        score=float(res["response"]==q.answer); reason=""
    else:
        score,reason=judge_score(q,res["response"],Counters())  
    srcs=[]
    for e in res["evidence"]:
        s=e.metadata.get("source_file") or e.path or e.name
        if s and s not in srcs: srcs.append(s)
    return {"config":config_name(cfg),"key":config_key(cfg),"qid":q.qid,
            "response":res["response"],"gold":q.answer,"score":round(score,3),"judge":reason,
            "correct":int(score>=0.5),"time":round(dt,3),
            "in_tok":c.in_tok,"out_tok":c.out_tok,"calls":c.calls,
            "steps":res["steps"],"backtracks":res["backtracks"],
            "path":res["path"],"sources":", ".join(srcs[:3]),"difficulty":q.difficulty}

def run_config(cfg,questions,max_q=None,verbose=False):
    reset_vcaches(); qs=questions[:max_q] if max_q else questions; rows=[]; key=config_key(cfg)
    for q in qs:
        ck=f"{key}::{q.qid}"
        if ck in CACHE: rec=CACHE[ck]
        else: rec=_eval_one(cfg,q); CACHE[ck]=rec; save_cache()
        rows.append(rec)
    return rows

def _print_diag(q,rec):
    o=q.options
    print("  "+"─"*86)
    print(f"  question: {q.stem}")
    for L in "ABCD": print(f"    {L}. {o[L]}")
    print(f"  correct: {rec['gold']}. {o[rec['gold']]}")
    print()
    print("  agent response:")
    print("    "+(rec["response"] or "(empty)").replace("\n","\n    "))
    print(f"  files reached: {rec['sources'] or '— none —'}")
    print(f"  full path: {rec['path']}  ({rec['steps']} steps, {rec['backtracks']} backtracks)")
    print()
    print(f"  judge: {rec['score']:.2f} — {rec.get('judge') or 'no justification returned'}")

def run_sample(questions, n=2, cfg=None, seed=0):
    cfg=_replace(cfg or Config(), nav_includes_options=False)    
    reset_vcaches()
    sample=_random.Random(seed).sample(questions, min(n,len(questions)))
    print(f"sampling {len(sample)} questions under: {config_name(cfg)}")
    rows=[]; tot=0.0
    for idx,q in enumerate(sample,1):
        print("\n"+"="*90)
        print(f"[{idx}/{len(sample)}]  {q.qid}")
        rec=_eval_one(cfg,q,live=True)
        rows.append(rec); tot+=rec["score"]
        _print_diag(q,rec)
    print("\n"+"="*90+f"\nmean judge score over {len(sample)} questions: {tot/len(sample):.3f}")
    return rows

def check_leaf_content(root=None):
    root=root or ROOT
    leaves=[]; stack=[root]
    while stack:
        n=stack.pop()
        if n.node_type=="chunk": leaves.append(n)
        else: stack.extend(n.children)
    total=max(len(leaves),1)
    withc=[len((l.content or "").strip()) for l in leaves if (l.content or "").strip()]
    frac=len(withc)/total
    print(f"leaf nodes: {len(leaves)}")
    print(f"  with raw content: {len(withc)} ({100*frac:.0f}%) | summary only: {len(leaves)-len(withc)}")
    if withc:
        withc.sort(); print(f"  raw content length: median {withc[len(withc)//2]} chars, max {max(withc)}")
    if frac<0.2:
        print("VERDICT: leaves are essentially summary-only, so the agent answers from lossy summaries; "
              "this likely caps accuracy regardless of navigation. rebuilding the tree to keep raw chunk text would help most.")
    elif frac<0.8:
        print("VERDICT: only some leaves carry raw text; check whether the docs you actually query are among the ones missing it.")
    else:
        print("VERDICT: leaves carry raw text, so the evidence is there; low accuracy points at navigation or the answer step, not the tree.")
    return {"leaves":len(leaves),"with_content":len(withc),"fraction":frac}

def run_diagnosis(questions, n=30, cfg=None, seed=0, verbose=True):
    cfg=_replace(cfg or Config(), nav_includes_options=False)
    reset_vcaches()
    sample=_random.Random(seed).sample(questions, min(n,len(questions)))
    print(f"diagnosis on {len(sample)} questions under: {config_name(cfg)}\n"+"-"*78)
    rows=[]; ev_sum=0.0; rp_sum=0.0; miss_ret=0; miss_ans=0; ok=0
    for q in sample:
        res=run_agent(q,cfg,Counters(),trace=False,live=False)
        es,er=judge_evidence(q,res["evidence"],Counters())
        rs,rr=judge_score(q,res["response"],Counters())
        if es<0.5: tag="RETRIEVAL MISS"; miss_ret+=1     
        elif rs<0.5: tag="ANSWER MISS"; miss_ans+=1       
        else: tag="ok"; ok+=1
        ev_sum+=es; rp_sum+=rs
        srcs=", ".join(dict.fromkeys(e.metadata.get("source_file") or e.path or e.name for e in res["evidence"]))
        rows.append({"qid":q.qid,"evidence":round(es,3),"response":round(rs,3),"tag":tag,
                     "files":srcs,"resp":res["response"],"ev_reason":er,"rp_reason":rr})
        if verbose:
            print(f"  {q.qid:12s} evidence={es:.2f}  response={rs:.2f}  -> {tag:14s}  {srcs[:54] or '— none —'}")
    k=len(sample)
    print("-"*78)
    print(f"mean evidence-score: {ev_sum/k:.3f}    mean response-score: {rp_sum/k:.3f}")
    print(f"retrieval misses: {miss_ret}/{k} ({100*miss_ret/k:.0f}%)   "
          f"answer misses: {miss_ans}/{k} ({100*miss_ans/k:.0f}%)   ok: {ok}/{k} ({100*ok/k:.0f}%)")
    if miss_ret/k>=0.4:
        print("VERDICT: navigation is the bottleneck, the agent often never reaches the right document. "
              "beam or top-k navigation, query expansion, or best-first search are the things to build next.")
    elif miss_ans/k>=0.4:
        print("VERDICT: retrieval is mostly fine but the final answer step loses it; fix the answer prompt, not the traversal.")
    else:
        print("VERDICT: most questions are handled well; if the headline number still looks low, hand-check the judge, it may be harsh.")
    return rows

def run_grid(grid,questions,max_q=None,verbose=False):
    qs=questions[:max_q] if max_q else questions
    tasks=[(cfg,q) for cfg in grid for q in qs]
    done=sum(1 for cfg,q in tasks if f"{config_key(cfg)}::{q.qid}" in CACHE)   # already in the cache
    if done: print(f"resuming; {done}/{len(tasks)} runs already cached, the bar starts from there")
    bar=_bar(len(tasks),done,"eval")
    all_rows=[]; score_sum=0.0; n=0; last=None
    for cfg,q in tasks:
        if cfg is not last:
            reset_vcaches(); last=cfg                 
            bar.set_description((config_name(cfg)[:34]))
        ck=f"{config_key(cfg)}::{q.qid}"
        if ck in CACHE:
            rec=CACHE[ck]                              
        else:
            rec=_eval_one(cfg,q); CACHE[ck]=rec; save_cache(); bar.update(1)
        all_rows.append(rec); score_sum+=rec["score"]; n+=1
        bar.set_postfix(score=f"{score_sum/n:.2f}",last=f"{rec['score']}")
    bar.close()
    print(f"done; {n} results, overall mean score {score_sum/n:.3f} across all configs")
    return all_rows


def leaderboard(all_rows):
    df=pd.DataFrame(all_rows)
    g=df.groupby(["config","key"],sort=False)
    lb=g.agg(n=("score","size"),correct=("correct","sum"),
             accuracy=("score","mean"),
             mean_time=("time","mean"),std_time=("time","std"),
             mean_in=("in_tok","mean"),std_in=("in_tok","std"),
             mean_out=("out_tok","mean"),std_out=("out_tok","std"),
             total_in=("in_tok","sum"),total_out=("out_tok","sum"),
             mean_calls=("calls","mean")).reset_index()
    lb["accuracy_pct"]=(lb["accuracy"]*100).round(1)
    for c in ["mean_time","std_time","mean_in","std_in","mean_out","std_out","mean_calls"]:
        lb[c]=lb[c].fillna(0).round(2)
    return lb

def by_accuracy(lb):
    return lb.sort_values(["accuracy","mean_time"],ascending=[False,True]).reset_index(drop=True)

def by_speed(lb):
    return lb.sort_values(["mean_time","accuracy"],ascending=[True,False]).reset_index(drop=True)

def save_outputs(lb,all_rows,outdir="eval_cache"):
    d=Path(outdir); d.mkdir(exist_ok=True)
    cols=["config","accuracy_pct","correct","n","mean_time","std_time",
          "mean_in","mean_out","total_in","total_out","mean_calls","key"]
    by_accuracy(lb)[cols].to_csv(d/"leaderboard_by_accuracy.csv",index=False)
    by_speed(lb)[cols].to_csv(d/"leaderboard_by_speed.csv",index=False)
    pd.DataFrame(all_rows).to_csv(d/"per_question.csv",index=False)
    print(f"wrote {d/'leaderboard_by_accuracy.csv'}, leaderboard_by_speed.csv and per_question.csv")

print("eval harness ready; load_questions, run_grid, leaderboard, by_accuracy, by_speed")

eval harness ready; load_questions, run_grid, leaderboard, by_accuracy, by_speed


/opt/homebrew/Cellar/jupyterlab/4.5.7_1/libexec/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
import heapq

LEAF_EMB_FILE=Path("nav_cache/leaf_embeddings.json")

def collect_leaves(root=None):
    root=root or ROOT; leaves=[]; st=[root]
    while st:
        n=st.pop()
        if n.node_type=="chunk": leaves.append(n)
        else: st.extend(n.children)
    return leaves

def build_leaf_index(root=None, use_content=False):
    leaves=collect_leaves(root)
    cache={}
    if LEAF_EMB_FILE.exists(): cache=json.loads(LEAF_EMB_FILE.read_text())
    todo=[l for l in leaves if l.node_id not in cache]
    print(f"{len(leaves)} leaves; {len(cache)} already embedded, {len(todo)} to go")
    if todo:
        bar=_bar(len(leaves),len(leaves)-len(todo),"embed leaves"); cnt=Counters()
        for i,l in enumerate(todo,1):
            txt=((l.content or l.summary) if use_content else (l.summary or l.content) or "")[:2000]
            v=embed(txt,cnt)
            if v is not None: cache[l.node_id]=v
            bar.update(1)
            if i%200==0:
                LEAF_EMB_FILE.parent.mkdir(exist_ok=True); LEAF_EMB_FILE.write_text(json.dumps(cache))
        bar.close()
        LEAF_EMB_FILE.parent.mkdir(exist_ok=True); LEAF_EMB_FILE.write_text(json.dumps(cache))
    ids=[l.node_id for l in leaves if l.node_id in cache]
    M=np.array([cache[i] for i in ids],dtype=float)
    M=M/np.clip(np.linalg.norm(M,axis=1,keepdims=True),1e-9,None)
    return ids, M, {l.node_id:l for l in leaves}

def embed_recall(q, ids, M, byid, k, counter):
    qv=embed(q.stem,counter)
    if qv is None: return 0.0, []
    qv=np.array(qv,dtype=float); qv=qv/max(np.linalg.norm(qv),1e-9)
    top=np.argsort(-(M@qv))[:k]
    leaves=[byid[ids[i]] for i in top]
    es,_=judge_evidence(q,leaves,Counters())
    return es, leaves

def closed_book(q, counter):
    resp=llm("answer this question in 1-3 sentences using your own knowledge.\n\nQUESTION: "+q.stem,
             Config(),counter,num_predict=320,temperature=0)
    rs,_=judge_score(q,resp,Counters())
    return rs, resp

def run_ceiling(questions, n=30, k=5, seed=0, agentic_evidence=None):
    sample=_random.Random(seed).sample(questions, min(n,len(questions)))
    ids,M,byid=build_leaf_index()
    cb=0.0; er=0.0; rows=[]
    bar=_bar(len(sample),0,"ceiling")
    for q in sample:
        c=Counters()
        cbs,cresp=closed_book(q,c)
        es,leaves=embed_recall(q,ids,M,byid,k,c)
        cb+=cbs; er+=es
        rows.append({"qid":q.qid,"closed_book":round(cbs,3),"embed_recall":round(es,3),
                     "top_files":", ".join(dict.fromkeys((l.metadata.get("source_file") or l.name) for l in leaves))[:64]})
        bar.update(1)
    bar.close()
    K=len(sample); cb/=K; er/=K
    print("\n"+"="*70)
    print(f"closed-book response-score   (base model, no retrieval) : {cb:.3f}")
    print(f"embedding recall@{k} evidence  (dense-retrieval ceiling)  : {er:.3f}")
    if agentic_evidence is not None:
        print(f"agentic TreeRAG evidence     (your best navigator)      : {agentic_evidence:.3f}")
    print("="*70)
    _interpret(cb, er, agentic_evidence)
    return rows, {"closed_book":cb, "embed_ceiling":er}

def _interpret(cb, ceiling, agentic):
    a = agentic if agentic is not None else 0.20
    if ceiling >= a + 0.15:
        print("READING: the answer IS findable from your leaf summaries, but the agentic tree-walk is losing it. "
              "the lever is the navigation CONTEXT, not the search strategy: give each routing decision more signal "
              "(show previews of what is under each child, use llm group summaries, or flatten the tree so fewer "
              "decisions compound). worth another iteration before vector search.")
    else:
        print("READING: even a perfect similarity search over your leaf summaries scores about the same as the agent, "
              "so the discriminative signal is not in the summaries. no navigator change can fix this. the lever is "
              "UPSTREAM: re-summarise leaves with more specific detail or index raw chunk text. if the ceiling is low "
              "in absolute terms too, the corpus may simply not contain single-leaf answers to these questions.")
    if cb >= 0.30:
        print(f"NOTE: closed-book already scores {cb:.2f}, so a chunk of response-score is the base model's prior "
              "knowledge, not your retrieval. weight EVIDENCE-score over response-score when you compare to vector search.")

print("ceiling diagnostic ready; run_ceiling(questions, n=30, k=5)")

ceiling diagnostic ready; run_ceiling(questions, n=30, k=5)


In [8]:
load_cache()
questions = load_questions(QUESTIONS_FILE, exclude_words=EXCLUDE_WORDS)
print(f"loaded {len(questions)} questions")

results cache: 0 stored runs
  sheet 'General QMS': kept 47
  sheet 'Genomics': kept 115
  sheet 'Tissue Portal': kept 53
  sheet 'GSI': kept 0
excluded 86 questions containing ['which']
parsed 129 questions total from eval_questions.xlsx
loaded 129 questions


In [9]:
rows, summary = run_ceiling(questions, n=30, k=5, seed=0, agentic_evidence=0.227)
import pandas as pd
pd.DataFrame(rows)

95458 leaves; 0 already embedded, 95458 to go


ceiling: 100%|██████████████████████████████████████████████████████| 30/30 [05:54<00:00, 11.81s/q]


closed-book response-score   (base model, no retrieval) : 0.477
embedding recall@5 evidence  (dense-retrieval ceiling)  : 0.355
agentic TreeRAG evidence     (your best navigator)      : 0.227
READING: even a perfect similarity search over your leaf summaries scores about the same as the agent, so the discriminative signal is not in the summaries. no navigator change can fix this. the lever is UPSTREAM: re-summarise leaves with more specific detail or index raw chunk text. if the ceiling is low in absolute terms too, the corpus may simply not contain single-leaf answers to these questions.
NOTE: closed-book already scores 0.48, so a chunk of response-score is the base model's prior knowledge, not your retrieval. weight EVIDENCE-score over response-score when you compare to vector search.


,qid,closed_book,embed_recall,top_files
0,TP-2,0.00,0.40,folders/Quality SOPs and Worksheets/Laboratory...
1,TP-22,0.15,0.90,folders/Management/Accreditation/CAP/CAP Check...
2,QMS-7,0.85,0.15,folders/Quality SOPs and Worksheets/Personnel ...
3,GEN-15,0.30,0.00,folders/Lab_Quality_Documents/Checklists/CAP/M...
4,TP-50,1.00,1.00,folders/Technical SOPs and Worksheets/Dual Ext...
5,GEN-64,0.00,0.00,folders/Lab_Quality_Documents/Equipment User M...
6,GEN-55,0.88,0.35,folders/Lab_Quality_Documents/QA Information S...
7,GEN-39,0.60,1.00,folders/Lab_Quality_Documents/Equipment User M...
8,TP-43,0.00,0.00,folders/Technical SOPs and Worksheets/Sample A...
9,TP-11,0.85,0.18,folders/Lab_Quality_Documents/Equipment User M...
